# RAG (Retrieval-Augmented Generation) — Step-by-Step Tutorial
### TIES 4911 (2026) — MiniProject Option 7

## Overview

This notebook walks through a complete RAG (Retrieval-Augmented Generation) pipeline for a **Deep Learning Knowledge Base**.

### What is RAG?
RAG is a technique that enhances Large Language Models (LLMs) by connecting them to external knowledge sources. Instead of relying solely on the model's parametric knowledge (which can be outdated or hallucinated), RAG:
1. **Retrieves** relevant information from a document database
2. **Augments** the LLM prompt with the retrieved context
3. **Generates** a grounded, factual answer

### Architecture
```
┌─────────────────────────────────────────────────────────────────┐
│                     INDEXING (offline)                          │
│  Documents → Chunking → Embedding → Vector Store (ChromaDB)    │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     QUERYING (online)                           │
│  Question → Embedding → Similarity Search → Top-K Chunks       │
│           → Prompt with Context → LLM → Answer                 │
└─────────────────────────────────────────────────────────────────┘
```

### Tech Stack
| Component | Tool |
|-----------|------|
| Document loading | LangChain (PyPDFLoader, TextLoader) |
| Text splitting | RecursiveCharacterTextSplitter |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector store | ChromaDB (local, persistent) |
| LLM | Ollama (local) or OpenAI GPT |
| Orchestration | LangChain |
| UI | Gradio |

## Step 0 — Installation

Install dependencies (run once):

In [ ]:
# Uncomment to install
# !pip install -r requirements.txt

## Step 1 — Load Documents

Load text/PDF files from the `data/documents/` folder.
Each file becomes one or more `Document` objects with text content and metadata.

In [ ]:
import sys
sys.path.insert(0, '.')  # ensure imports from project root

from src.document_processor import load_documents

documents = load_documents('./data/documents')
print(f'Loaded {len(documents)} document(s)')
print('\nFirst document preview:')
print(documents[0].page_content[:500])
print('\nMetadata:', documents[0].metadata)

## Step 2 — Chunk Documents

Long documents must be split into smaller chunks before embedding.

**Why?**
- Embedding models have a token limit (~512 tokens for MiniLM)
- Smaller chunks = more precise retrieval (exact passage, not whole document)
- Overlap preserves context at chunk boundaries

We use `RecursiveCharacterTextSplitter` which tries to split on paragraph breaks (`\n\n`) first, then sentences, then words.

In [ ]:
from src.document_processor import split_documents

chunks = split_documents(documents)

print(f'Number of chunks: {len(chunks)}')
print(f'\nExample chunk:')
print(chunks[5].page_content)
print(f'\nChunk length: {len(chunks[5].page_content)} characters')

## Step 3 — Create Embeddings

An **embedding** is a dense vector representation of text.
Semantically similar texts have similar (close) embeddings in the vector space.

We use `all-MiniLM-L6-v2` — a small (80MB) but powerful sentence-transformers model:
- 384-dimensional vectors
- Excellent semantic similarity performance
- Fast on CPU

In [ ]:
from src.vector_store import get_embeddings
import numpy as np

embeddings = get_embeddings()

# Embed a sample text
sample_text = 'How does self-attention work in Transformers?'
vector = embeddings.embed_query(sample_text)

print(f'Embedding dimension: {len(vector)}')
print(f'First 10 values: {np.round(vector[:10], 4)}')

# Show that similar sentences have similar embeddings
v1 = np.array(embeddings.embed_query('What is attention mechanism?'))
v2 = np.array(embeddings.embed_query('How does self-attention work?'))
v3 = np.array(embeddings.embed_query('What is the capital of France?'))

cos_sim = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f'\nCosine similarity (related questions): {cos_sim(v1, v2):.3f}')
print(f'Cosine similarity (unrelated questions): {cos_sim(v1, v3):.3f}')

## Step 4 — Build the Vector Store (ChromaDB)

We embed all chunks and store them in ChromaDB — a local, persistent vector database.
ChromaDB enables fast approximate nearest-neighbor search to find the most relevant chunks.

In [ ]:
from src.vector_store import build_vector_store

vector_store = build_vector_store(chunks)
print(f'Vector store built with {vector_store._collection.count()} chunks indexed')

## Step 5 — Retrieve Relevant Chunks

Given a user query, we:
1. Embed the query using the same model
2. Compute cosine similarity between the query vector and all stored chunk vectors
3. Return the top-K most similar chunks

In [ ]:
from src.vector_store import retrieve

query = 'How does the Transformer attention mechanism work?'
results = retrieve(vector_store, query, k=3)

print(f'Query: "{query}"')
print(f'\nTop {len(results)} retrieved chunks:\n')
for i, (doc, score) in enumerate(results, 1):
    source = doc.metadata.get('source_file', 'unknown')
    print(f'--- Chunk {i} (score: {score:.3f}, source: {source}) ---')
    print(doc.page_content[:300])
    print()

## Step 6 — Generate an Answer with LLM

The retrieved chunks are inserted into the LLM prompt as context.
The LLM is instructed to answer ONLY from this context, preventing hallucinations.

**Note:** Set your LLM backend in `.env`:
- `LLM_BACKEND=ollama` (local, free — requires Ollama running with a model)
- `LLM_BACKEND=openai` (requires OpenAI API key)
- `LLM_BACKEND=retrieval_only` (no LLM — returns retrieved passages directly)

In [ ]:
from src.llm_interface import get_llm, generate_answer

llm = get_llm()  # returns None in retrieval_only mode
result = generate_answer(query, results, llm)

print('ANSWER:')
print('=' * 60)
print(result['answer'])
print('\nSources:', result['sources'])

## Step 7 — Full Pipeline Demo

Using the high-level `RAGPipeline` class to ask multiple questions:

In [ ]:
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
pipeline.initialize()

test_questions = [
    'What is Retrieval-Augmented Generation?',
    'How does YOLO detect objects?',
    'What is the vanishing gradient problem?',
    'How do autoencoders work for anomaly detection?',
    'What is the difference between VAE and GAN?',
]

for q in test_questions:
    result = pipeline.query(q)
    print(f'Q: {q}')
    print(f'A: {result["answer"][:300]}...')
    print(f'Sources: {result["sources"]}')
    print('-' * 70)

## Step 8 — Retrieval Quality Analysis

Let's visualize how well the retrieval works by examining relevance scores:

In [ ]:
import matplotlib.pyplot as plt

test_q = 'What is self-attention in Transformers?'
results = retrieve(vector_store, test_q, k=8)

labels = [f"Chunk {i+1}\n({d.metadata.get('source_file','?')[:15]})" 
          for i, (d, _) in enumerate(results)]
scores = [s for _, s in results]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels[::-1], scores[::-1], color='steelblue')
ax.set_xlabel('Cosine Similarity Score')
ax.set_title(f'Retrieval Scores for:\n"{test_q}"')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Threshold=0.5')
ax.legend()
plt.tight_layout()
plt.savefig('retrieval_scores.png', dpi=150)
plt.show()
print('Saved: retrieval_scores.png')

## Step 9 — Adding Your Own Documents

To extend the knowledge base with your own PDFs or text files:

1. Place your files in `data/documents/`
2. Re-run ingestion:
   ```bash
   python ingest.py
   ```
3. Query as usual — the new documents are automatically indexed

Supported formats: `.pdf`, `.txt`, `.md`

**Tips for best results:**
- Use documents with clear, structured text
- Adjust `CHUNK_SIZE` (default 500) for your domain: smaller = more precise, larger = more context
- If using OpenAI, GPT-4 gives much better answers than GPT-3.5

## Step 10 — Launch the Web UI

Run the Gradio web interface for interactive querying:
```bash
python app.py
```
Then open http://localhost:7860 in your browser.

## Summary

This project demonstrates a complete RAG pipeline:

| Phase | What happens | Tools |
|-------|-------------|-------|
| **Ingestion** | Load documents | LangChain loaders |
| | Split into chunks | RecursiveCharacterTextSplitter |
| | Embed chunks | sentence-transformers |
| | Store vectors | ChromaDB |
| **Querying** | Embed query | same embedding model |
| | Similarity search | ChromaDB cosine similarity |
| | Build prompt | LangChain PromptTemplate |
| | Generate answer | Ollama / OpenAI LLM |

### Key benefits of RAG over pure LLM:
- **No hallucinations** — answers grounded in real documents
- **Up-to-date** — add new documents without retraining
- **Domain-specific** — specialized knowledge without fine-tuning cost
- **Transparent** — can show which sources were used
- **Cost-effective** — cheaper than full fine-tuning